# Project Part 2: Parameter-Efficient Fine-Tuning of CLIP for Image–Text Retrieval

**Total Marks: 20**

**Group Name:** __________________

**Student Name (Student ID):**

1. __________________ (__________________)
2. __________________ (__________________)
3. __________________ (__________________)
4. __________________ (__________________)

## Objective

In Part 1, CLIP was a **fixed** feature extractor. In Part 2 you will **adapt** CLIP to image–text retrieval on **Flickr8k** with two **parameter-efficient fine-tuning (PEFT)** methods, and compare them with the zero-shot model.

**Tasks**

- Freeze CLIP and set up projection-only PEFT.
- Load and split Flickr8k (train / validation / test); cache frozen encoder outputs.
- Implement the **InfoNCE** and **triplet** contrastive losses and **Recall@K**.
- Implement a **LoRA** layer from scratch (initialisation, forward pass, weight merging).
- Write the training step, run the experiments, and analyse the results.

**How to answer:** write all answers in the **"Your answer"** cells. Run the notebook top to bottom before submitting so that all outputs are saved.

**Compute budget:** caching the frozen features takes a few minutes (once); projection-only runs take seconds; each LoRA run (5 epochs) took ≈ 100 s on an Apple-silicon laptop (MPS) and will take several times longer on a CPU-only laptop (reduce `LORA_EPOCHS` to 3 if needed). No GPU is needed.

**Requires:** Python 3.12, `torch` ≥ 2.4, `transformers` ≥ 5.0 (also 4.4x), `datasets` ≥ 3.0, `matplotlib`, `tensorboard`. Optional: `peft`.

**Marks:** Step 2 [1] · Step 3 [1] · Step 4 [2] · Step 5 [2] · Step 6 [1] · Step 7 [3] · Step 8 [2] · Step 9 analysis [8]

## Step 1: Setup and Imports (Provided)

In [ ]:
import os
import gc
import time
import random
from collections import defaultdict
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
MODEL_STR = "openai/clip-vit-base-patch32"
CACHE_DIR = "./cache_part2"
os.makedirs(CACHE_DIR, exist_ok=True)
print("Device:", device)

## Step 2: Load and Explore the Pre-trained CLIP [1 Mark]

**Your Task:** load the CLIP model (with `attn_implementation="sdpa"`, moved to `device`) and its processor, as in Part 1.

> A `LOAD REPORT` listing `position_ids` as `UNEXPECTED` and a note about the *fast* image processor are harmless.

In [ ]:
# ================================
# TODO: YOUR CODE GOES HERE (1 Mark)
# ================================
model = None
processor = None

### 2.1 Parameter counts (Provided)

In [ ]:
def count_params(module: nn.Module) -> Tuple[int, int, float]:
    """Return (total, trainable, % trainable)."""
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable, 100.0 * trainable / max(1, total)

def param_breakdown(model: nn.Module):
    """Print total / trainable parameters per top-level component of CLIP."""
    groups = defaultdict(lambda: [0, 0])
    for name, p in model.named_parameters():
        parts = name.split(".")
        key = parts[1] if parts[0] == "clip" else parts[0]          # also works for the wrapper below
        if "lora_" in name:
            key = f"{key} (LoRA)"
        groups[key][0] += p.numel()
        groups[key][1] += p.numel() if p.requires_grad else 0
    print(f"{'component':<26}{'total':>14}{'trainable':>14}")
    for key, (tot, tr) in groups.items():
        print(f"{key:<26}{tot:>14,}{tr:>14,}")
    total, trainable, pct = count_params(model)
    print(f"{'ALL':<26}{total:>14,}{trainable:>14,}   ({pct:.2f}% trainable)")

param_breakdown(model)

## Step 3: PEFT Method 1: Projection-only Fine-tuning [1 Mark]

CLIP maps each encoder output into the shared 512-d space with a single linear layer: `visual_projection` (768 → 512) and `text_projection` (512 → 512). Projection-only fine-tuning **freezes everything else**, including `logit_scale`.

**Your Task:** complete `setup_projection_peft`.

In [ ]:
def setup_projection_peft(model: CLIPModel) -> CLIPModel:
    """Freeze all CLIP parameters, then unfreeze only `visual_projection` and `text_projection`."""
    # ================================
    # TODO: YOUR CODE GOES HERE (1 Mark)
    # ================================
    raise NotImplementedError("Please implement `setup_projection_peft`.")

model = setup_projection_peft(model)
param_breakdown(model)

In [ ]:
# === Auto-check: Step 3 ===
assert all(not p.requires_grad for p in model.vision_model.parameters()), "vision_model must be frozen"
assert all(not p.requires_grad for p in model.text_model.parameters()), "text_model must be frozen"
assert not model.logit_scale.requires_grad, "logit_scale must be frozen"
assert all(p.requires_grad for p in model.visual_projection.parameters()), "visual_projection must be trainable"
assert all(p.requires_grad for p in model.text_projection.parameters()), "text_projection must be trainable"
assert count_params(model)[1] == 655_360, f"expected 655,360 trainable parameters, got {count_params(model)[1]:,}"
print("✅ Step 3 check passed.")

## Step 4: Data Preparation (Flickr8k) [2 Marks]

We use `Naveengo/flickr8k` from the Hugging Face Hub: 8,091 photos, each paired with one caption. We split it into **6,000 train / 1,000 validation / 1,091 test** pairs.

- **Train** is used to update the weights.
- **Validation** is used to choose hyper-parameters (learning rate, epoch, LoRA rank).
- **Test** is used **only once**, for the final comparison table. Choosing settings on the test set would make the reported numbers optimistic.

**Caching frozen features.** The vision encoder is frozen in *every* method, so its output for an image never changes. We compute it once and save it to disk; training then only runs the small trainable parts. This is the main reason PEFT is cheap. The same holds for the text encoder in projection-only fine-tuning, so we cache its outputs too. (LoRA changes the text encoder, so LoRA runs execute the text encoder live.)

### 4.1 Load and split the dataset [1 Mark]

In [ ]:
def load_and_split_dataset(n_train: int = 6000, n_val: int = 1000, seed: int = 42):
    """
    Returns:
        all_data:   the full Hugging Face split ("train" split of "Naveengo/flickr8k")
        train_idx, val_idx, test_idx: disjoint lists of row indices

    Instructions:
    1. all_data = load_dataset("Naveengo/flickr8k", split="train")
    2. Create the list [0, 1, ..., N-1] and shuffle it with random.Random(seed).shuffle(...)
       (a local generator, so the split does not depend on other random calls).
    3. The first n_train indices are train, the next n_val are validation, the rest are test.
    """
    # ================================
    # TODO: YOUR CODE GOES HERE (1 Mark)
    # ================================
    raise NotImplementedError("Please implement `load_and_split_dataset`.")

In [ ]:
all_data, train_idx, val_idx, test_idx = load_and_split_dataset()
all_captions = list(all_data["text"])          # text column only: no image decoding
print(f"Pairs: {len(all_data)} | train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")
assert len(set(train_idx) | set(val_idx) | set(test_idx)) == len(all_data), "splits must cover all rows"
assert not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx)), "splits must be disjoint"
print("Example caption:", all_captions[train_idx[0]])
all_data[train_idx[0]]["image"].resize((224, 224))

### 4.2 Cache the frozen encoder outputs (Provided)

In [ ]:
@torch.no_grad()
def cache_vision_features(model, processor, hf_dataset, path, batch_size=64) -> torch.Tensor:
    """Pooled output of the frozen vision encoder (before visual_projection), shape (N, 768)."""
    if os.path.exists(path):
        return torch.load(path)
    model.eval()
    feats = []
    for start in tqdm(range(0, len(hf_dataset), batch_size), desc="Caching image features"):
        images = [im.convert("RGB") for im in hf_dataset[start:start + batch_size]["image"]]
        pixel_values = processor(images=images, return_tensors="pt")["pixel_values"].to(device)
        feats.append(model.vision_model(pixel_values=pixel_values).pooler_output.float().cpu())
    feats = torch.cat(feats)
    torch.save(feats, path)
    return feats

@torch.no_grad()
def cache_text_features(model, tokenizer, captions, path, batch_size=256) -> torch.Tensor:
    """Pooled output of the frozen text encoder (before text_projection), shape (N, 512)."""
    if os.path.exists(path):
        return torch.load(path)
    model.eval()
    feats = []
    for start in tqdm(range(0, len(captions), batch_size), desc="Caching text features"):
        tok = tokenizer(captions[start:start + batch_size], padding=True, truncation=True,
                        max_length=77, return_tensors="pt").to(device)
        feats.append(model.text_model(**tok).pooler_output.float().cpu())
    feats = torch.cat(feats)
    torch.save(feats, path)
    return feats

vision_feats = cache_vision_features(model, processor, all_data, os.path.join(CACHE_DIR, "vision_pooled.pt"))
text_feats = cache_text_features(model, processor.tokenizer, all_captions, os.path.join(CACHE_DIR, "text_pooled.pt"))
print("vision_feats:", tuple(vision_feats.shape), "| text_feats:", tuple(text_feats.shape))

### 4.3 Dataset and DataLoaders [1 Mark]

**Your Task:** complete `FlickrFeatureDataset.__getitem__`. Position `idx` refers to `self.indices[idx]`, a row of the full dataset. The provided `collate` function tokenises the captions of a batch with **dynamic padding**, i.e. to the longest caption in the batch rather than to 77 tokens. This is much faster for the LoRA runs.

In [ ]:
class FlickrFeatureDataset(Dataset):
    def __init__(self, indices, vision_feats, text_feats, captions):
        self.indices = list(indices)
        self.vision_feats = vision_feats
        self.text_feats = text_feats
        self.captions = captions

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Dict:
        """
        Return a dict with keys:
            "vision_feats": FloatTensor (768,)  cached vision feature of row self.indices[idx]
            "text_feats":   FloatTensor (512,)  cached text feature of the same row
            "caption":      str                 its caption
            "row":          int                 the row index itself
        """
        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `__getitem__`.")

In [ ]:
# (Provided) collate function, loaders and sanity check
tokenizer = processor.tokenizer

def collate(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    tok = tokenizer([b["caption"] for b in batch], padding=True, truncation=True,
                    max_length=77, return_tensors="pt")
    return {
        "vision_feats": torch.stack([b["vision_feats"] for b in batch]),
        "text_feats": torch.stack([b["text_feats"] for b in batch]),
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "row": torch.tensor([b["row"] for b in batch]),
    }

def to_device(batch: Dict, device) -> Dict:
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

train_ds = FlickrFeatureDataset(train_idx, vision_feats, text_feats, all_captions)
val_ds = FlickrFeatureDataset(val_idx, vision_feats, text_feats, all_captions)
test_ds = FlickrFeatureDataset(test_idx, vision_feats, text_feats, all_captions)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=collate, generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, collate_fn=collate)

batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"{k:<15}{tuple(v.shape)}")
assert batch["vision_feats"].shape == (BATCH_SIZE, 768) and batch["text_feats"].shape == (BATCH_SIZE, 512)
assert batch["input_ids"].shape == batch["attention_mask"].shape and batch["input_ids"].shape[1] <= 77
print("✅ Step 4 check passed.")

## Step 5: Contrastive Losses [2 Marks: InfoNCE 1, Triplet 1]

First, the provided wrapper `RetrievalCLIP` turns a batch into embeddings. It uses the cached text features when the text encoder is frozen, and runs the text encoder live when it has trainable (LoRA) parameters.

Both losses receive **un-normalised** embeddings `img` and `txt` of shape (B, 512), where row *i* of `img` matches row *i* of `txt`. **L2-normalise them inside the loss.** Let S = img · txtᵀ be the (B, B) cosine-similarity matrix.

- **InfoNCE (symmetric, as in CLIP):** logits = S / τ. Loss = ½ [CE(logits, labels) + CE(logitsᵀ, labels)] with labels = [0, 1, …, B−1]. Default τ = 0.01, which matches CLIP's learned `logit_scale` ≈ 100.
- **Triplet with the hardest in-batch negative (VSE++):** for each image, the hardest negative caption is the most similar *non-matching* caption in the batch, and vice versa. Loss = mean over i of [m − S_ii + max_{j≠i} S_ij]₊ + [m − S_ii + max_{j≠i} S_ji]₊, with margin m = 0.2.

In [ ]:
class RetrievalCLIP(nn.Module):
    """(Provided) Maps a batch to (un-normalised) 512-d image and text embeddings."""
    def __init__(self, clip: CLIPModel):
        super().__init__()
        self.clip = clip
        self.text_frozen = not any(p.requires_grad for p in clip.text_model.parameters())

    def image_embeds(self, batch) -> torch.Tensor:
        return self.clip.visual_projection(batch["vision_feats"])

    def text_embeds(self, batch) -> torch.Tensor:
        if self.text_frozen:                   # projection-only / zero-shot: use the cache
            pooled = batch["text_feats"]
        else:                                  # LoRA: run the (adapted) text encoder
            pooled = self.clip.text_model(input_ids=batch["input_ids"],
                                          attention_mask=batch["attention_mask"]).pooler_output
        return self.clip.text_projection(pooled)

# Sanity check: cached text features == running the frozen text encoder live
with torch.no_grad():
    b = to_device(next(iter(val_loader)), device)
    live = model.text_model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).pooler_output
    assert torch.allclose(live.float(), b["text_feats"], atol=1e-3), "cached text features do not match"
print("✅ Cached features match the live encoder.")

In [ ]:
def info_nce_loss(img: torch.Tensor, txt: torch.Tensor, temperature: float = 0.01) -> torch.Tensor:
    """Symmetric InfoNCE (CLIP loss). Returns a scalar tensor."""
    # ================================
    # TODO: YOUR CODE GOES HERE (1 Mark)
    # ================================
    raise NotImplementedError("Please implement `info_nce_loss`.")


def triplet_loss(img: torch.Tensor, txt: torch.Tensor, margin: float = 0.2) -> torch.Tensor:
    """Bidirectional triplet loss with the hardest in-batch negative. Returns a scalar tensor."""
    # ================================
    # TODO: YOUR CODE GOES HERE (1 Mark)
    # Hint: mask the diagonal with -inf (masked_fill) before taking the max.
    # ================================
    raise NotImplementedError("Please implement `triplet_loss`.")

In [ ]:
# === Auto-check: Step 5 ===
torch.manual_seed(0)
a, b = torch.randn(8, 512), torch.randn(8, 512)
for fn in (info_nce_loss, triplet_loss):
    out = fn(a, b)
    assert torch.is_tensor(out) and out.ndim == 0, f"{fn.__name__} must return a scalar tensor"
assert info_nce_loss(a, a).item() < 1e-3, "InfoNCE should be ~0 when every pair is perfectly matched"
assert abs(info_nce_loss(a, a, temperature=1e6).item() - np.log(8)) < 1e-3, "with huge τ, InfoNCE → log(B)"
assert triplet_loss(a, a).item() == 0.0, "triplet loss should be 0 when positives beat negatives by the margin"
o = torch.ones(8, 512)
assert abs(triplet_loss(o, o).item() - 0.4) < 1e-5, "all embeddings identical: loss = 2·margin"
print("✅ Step 5 check passed.")

## Step 6: Recall@K [1 Mark]

Given L2-normalised `img` and `txt` embeddings (N, 512) where row *i* of each forms a pair:

- **I2T R@K:** the fraction of images whose own caption is among the K most similar captions.
- **T2I R@K:** the fraction of captions whose own image is among the K most similar images.

*Hint:* with S = img · txtᵀ, the rank of the correct caption for image *i* is the number of captions with similarity **greater than** S_ii. The pair is retrieved within K if rank < K. This needs no loops.

In [ ]:
def recall_at_k(img: torch.Tensor, txt: torch.Tensor, k_values: Sequence[int] = (1, 5, 10)) -> Dict[str, float]:
    """Return {"I2T_R@1": ..., "I2T_R@5": ..., "I2T_R@10": ..., "T2I_R@1": ..., ...} as floats in [0, 1]."""
    # ================================
    # TODO: YOUR CODE GOES HERE (1 Mark)
    # ================================
    raise NotImplementedError("Please implement `recall_at_k`.")

In [ ]:
# (Provided) evaluation helpers
@torch.no_grad()
def encode_split(model: RetrievalCLIP, loader) -> Tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    imgs, txts = [], []
    for batch in loader:
        batch = to_device(batch, device)
        imgs.append(F.normalize(model.image_embeds(batch).float(), dim=-1))
        txts.append(F.normalize(model.text_embeds(batch).float(), dim=-1))
    return torch.cat(imgs), torch.cat(txts)

def evaluate(model: RetrievalCLIP, loader) -> Dict[str, float]:
    metrics = recall_at_k(*encode_split(model, loader))
    metrics["mean_R"] = float(np.mean(list(metrics.values())))   # mean of the six recalls
    return metrics

def fmt(m: Dict[str, float]) -> str:
    return "  ".join(f"{k}={v:.4f}" for k, v in m.items())

In [ ]:
# === Auto-check: Step 6 ===
e = torch.eye(5)
assert all(v == 1.0 for v in recall_at_k(e, e).values()), "perfect alignment must give recall 1"
r = recall_at_k(e, e[[1, 0, 2, 3, 4]], k_values=(1, 2))
assert abs(r["I2T_R@1"] - 0.6) < 1e-6 and abs(r["T2I_R@1"] - 0.6) < 1e-6 and r["I2T_R@2"] == 1.0
print("✅ Step 6 check passed.")

## Step 7: PEFT Method 2: LoRA (Low-Rank Adaptation) [3 Marks]

LoRA (Hu et al., 2021) freezes a pre-trained weight matrix and learns a **low-rank update**. In PyTorch notation (`nn.Linear` stores `weight` with shape (out, in) and computes y = x Wᵀ + b):

$$ y = x W^\top + b + \frac{\alpha}{r}\,(x A^\top) B^\top, \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r},\; r \ll \min(d_{in}, d_{out}). $$

(In the lecture convention, where W is stored as (inputs × outputs) and f = Wᵀa, PyTorch's `weight` is that matrix transposed.)

- **Parameters:** r·(d_in + d_out) instead of d_in·d_out. For a 512×512 attention projection with r = 8, that is 8,192 instead of 262,144 (3%).
- **Initialisation:** A random (Kaiming-uniform), **B = 0**, so the adapted layer equals the pre-trained layer at step 0 and training starts from the zero-shot model.
- **Scaling:** α/r keeps the update size comparable when r changes.
- **Merging:** after training, W ← W + (α/r)·B A gives a normal `nn.Linear` with **no extra inference cost**.

We add LoRA to the **query and value projections** (`q_proj`, `v_proj`) in every layer of CLIP's **text encoder**, the standard choice from the LoRA paper. The projections stay trainable as in Step 3. The vision encoder stays frozen (and cached) to keep the runs CPU-friendly.

**Your Task:** implement `LoRALinear`: `__init__` [1], `forward` [1], `merge` [1].

In [ ]:
import math

class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank update."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0, dropout: float = 0.05):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)          # the pre-trained weight stays frozen
        self.r = r
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)
        self.merged = False
        d_in, d_out = base.in_features, base.out_features
        factory = dict(device=base.weight.device, dtype=base.weight.dtype)
        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # Create trainable nn.Parameters `self.lora_A` with shape (r, d_in) and `self.lora_B` with shape (d_out, r)
        # (pass **factory). Initialise A with nn.init.kaiming_uniform_(..., a=math.sqrt(5)) and B with zeros.
        # ================================
        raise NotImplementedError("Please create lora_A and lora_B.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """base(x) + scaling * (dropout(x) @ Aᵀ) @ Bᵀ   (skip the LoRA branch once merged)."""
        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `forward`.")

    @torch.no_grad()
    def merge(self):
        """Fold the update into the frozen weight: W <- W + scaling * B @ A, then set self.merged = True."""
        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `merge`.")

In [ ]:
# === Auto-check: Step 7 ===
torch.manual_seed(0)
lin = nn.Linear(16, 8)
lora = LoRALinear(lin, r=4, alpha=8, dropout=0.0)
x = torch.randn(5, 16)
assert lora.lora_A.shape == (4, 16) and lora.lora_B.shape == (8, 4), "check the shapes of A and B"
assert lora.lora_A.requires_grad and lora.lora_B.requires_grad and not lin.weight.requires_grad
assert torch.allclose(lora(x), lin(x)), "with B = 0 the LoRA layer must equal the base layer"
nn.init.normal_(lora.lora_B)                       # pretend we trained B
y_before = lora(x)
assert not torch.allclose(y_before, lin(x)), "LoRA branch has no effect"
lora.merge()
assert torch.allclose(lora(x), y_before, atol=1e-5), "merged layer must give the same output"
print("✅ Step 7 check passed.")

### 7.1 Inject LoRA into CLIP (Provided)

In [ ]:
def inject_lora(module: nn.Module, targets=("q_proj", "v_proj"), r=8, alpha=16.0, dropout=0.05) -> int:
    """Replace every nn.Linear attribute named in `targets` (inside `module`) with a LoRALinear."""
    n = 0
    for parent in list(module.modules()):
        for name in targets:
            child = getattr(parent, name, None)
            if isinstance(child, nn.Linear):
                setattr(parent, name, LoRALinear(child, r=r, alpha=alpha, dropout=dropout))
                n += 1
    return n

def setup_lora_peft(model: CLIPModel, r=8, alpha=16.0, dropout=0.05) -> CLIPModel:
    """Projection-only PEFT + LoRA adapters on q_proj / v_proj of every text-encoder layer."""
    model = setup_projection_peft(model)
    n = inject_lora(model.text_model, r=r, alpha=alpha, dropout=dropout)
    print(f"Injected LoRA (r={r}, alpha={alpha}) into {n} linear layers of the text encoder")
    return model

def merge_lora(model: nn.Module) -> int:
    n = 0
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.merge()
            n += 1
    return n

def fresh_clip() -> CLIPModel:
    return CLIPModel.from_pretrained(MODEL_STR, attn_implementation="sdpa").to(device)

demo = setup_lora_peft(fresh_clip(), r=8)
param_breakdown(demo)
del demo; _ = gc.collect()

In [ ]:
# (Optional, provided) The same adapters with Hugging Face's `peft` library — compare the LoRA parameter count.
try:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                     target_modules=r".*text_model.*\.(q_proj|v_proj)")
    peft_model = get_peft_model(CLIPModel.from_pretrained(MODEL_STR), cfg)
    peft_model.print_trainable_parameters()      # LoRA only (projections not included)
    del peft_model; gc.collect()
except ImportError:
    print("`peft` not installed — optional (pip install peft).")

## Step 8: Trainer [2 Marks]

The trainer is provided except for the **training step** inside `train`. It evaluates on the **validation** set before training (epoch 0) and after every epoch, logs to TensorBoard (`tensorboard --logdir runs`) and keeps the per-epoch history. **Early stopping:** if validation mean recall has not improved for `patience` consecutive epochs, training stops; at the end the trainable weights of the best validation epoch are restored. `patience=None` runs all epochs (used in 9.3 and 9.4, where the full curves are the point of the experiment).

**Your Task:** implement the training step [2]: compute the image and text embeddings with `self.model.image_embeds(batch)` / `self.model.text_embeds(batch)`, compute `loss = loss_fn(img, txt)`, then zero the gradients, back-propagate and take an optimiser step.

In [ ]:
def build_optimizer(model: nn.Module, lr: float, lora_lr: float = None, weight_decay: float = 0.01):
    """(Provided) AdamW over trainable parameters only; LoRA parameters may use their own learning rate."""
    proj, lora = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (lora if "lora_" in name else proj).append(p)
    groups = [{"params": proj, "lr": lr}]
    if lora:
        groups.append({"params": lora, "lr": lora_lr if lora_lr is not None else lr})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


class RetrievalTrainer:
    def __init__(self, model: RetrievalCLIP, train_loader, val_loader, device, run_name: str):
        self.model = model.to(device)
        self.train_loader, self.val_loader = train_loader, val_loader
        self.device = device
        self.run_name = run_name
        self.writer = SummaryWriter(f"runs/{run_name}")
        self.history: List[Dict] = []
        self.best_score, self.best_epoch, self.best_state = -1.0, 0, None

    def _trainable_state(self):
        return {n: p.detach().clone() for n, p in self.model.named_parameters() if p.requires_grad}

    def _log(self, epoch: int, train_loss: float, val: Dict[str, float]):
        self.history.append({"epoch": epoch, "train_loss": train_loss, **val})
        if not np.isnan(train_loss):
            self.writer.add_scalar("Loss/train", train_loss, epoch)
        for k, v in val.items():
            self.writer.add_scalar(f"Val/{k}", v, epoch)
        improved = val["mean_R"] > self.best_score
        if improved:
            self.best_score, self.best_epoch, self.best_state = val["mean_R"], epoch, self._trainable_state()
        print(f"[{self.run_name}] epoch {epoch:>2}  train_loss={train_loss:.4f}  val: {fmt(val)}")
        return improved

    def train(self, epochs: int, loss_fn: Callable, lr: float, lora_lr: float = None,
              patience: Optional[int] = None):
        optimizer = build_optimizer(self.model, lr, lora_lr)
        self._log(0, float("nan"), evaluate(self.model, self.val_loader))
        start, epochs_without_improvement = time.time(), 0
        for epoch in range(1, epochs + 1):
            self.model.train()
            running, steps = 0.0, 0
            for batch in tqdm(self.train_loader, desc=f"epoch {epoch}/{epochs}", leave=False):
                batch = to_device(batch, self.device)
                # ================================
                # TODO: YOUR CODE GOES HERE (2 Marks): the training step
                # ================================
                raise NotImplementedError("Please implement the training step.")

                running += loss.item()
                steps += 1
            if self._log(epoch, running / steps, evaluate(self.model, self.val_loader)):
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if patience is not None and epochs_without_improvement >= patience:
                    print(f"[{self.run_name}] early stopping: no improvement for {patience} epochs")
                    break
        self.writer.close()
        self.model.load_state_dict(self.best_state, strict=False)      # early stopping
        print(f"[{self.run_name}] done in {time.time() - start:.0f}s; restored best epoch "
              f"{self.best_epoch} (val mean_R={self.best_score:.4f})")
        return self.history


def plot_histories(histories: Dict[str, List[Dict]], metric: str = "mean_R", title: str = ""):
    """(Provided) Left: training loss. Right: a validation metric. One line per run."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, hist in histories.items():
        ep = [h["epoch"] for h in hist]
        axes[0].plot(ep[1:], [h["train_loss"] for h in hist][1:], marker="o", label=name)
        axes[1].plot(ep, [h[metric] for h in hist], marker="o", label=name)
    axes[0].set(xlabel="epoch", ylabel="train loss", title="Training loss")
    axes[1].set(xlabel="epoch", ylabel=f"val {metric}", title=f"Validation {metric}")
    for ax in axes:
        ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()

## Step 9: Experiments and Analysis [8 Marks]

Run the cells in order. **Protocol:** every training run is evaluated on the **validation** set only; all choices (learning rate, epoch, LoRA rank) are made on validation. The **test** set is used **once**, in 9.7, for the selected model of each method.

### 9.1 Zero-shot baseline (Provided)

In [ ]:
results_val: Dict[str, Dict[str, float]] = {}   # best-epoch validation metrics of every run
histories: Dict[str, List[Dict]] = {}           # per-epoch history of every run
checkpoints: Dict[str, Tuple[Callable, Dict]] = {}   # how to rebuild each run + its trained weights

zero_shot = RetrievalCLIP(setup_projection_peft(fresh_clip())).to(device)
results_val["Zero-shot"] = evaluate(zero_shot, val_loader)
print("val:", fmt(results_val["Zero-shot"]))
del zero_shot; _ = gc.collect()

### 9.2 Projection-only: InfoNCE vs Triplet (Provided)

In [ ]:
PATIENCE = 3   # early-stopping patience (epochs without validation improvement)

def run(name, build_clip, loss_fn, epochs, lr, lora_lr=None, patience=PATIENCE):
    """(Provided) Train one configuration and return its best-epoch VALIDATION metrics.

    The test set is not touched. Only the small trainable weights are kept (in `checkpoints`),
    so the model can be rebuilt with `load_trained(name)` without keeping many CLIP copies in memory.
    """
    set_seed(42)
    model = RetrievalCLIP(build_clip())
    trainer = RetrievalTrainer(model, train_loader, val_loader, device, run_name=name)
    histories[name] = trainer.train(epochs=epochs, loss_fn=loss_fn, lr=lr, lora_lr=lora_lr, patience=patience)
    best = next(h for h in histories[name] if h["epoch"] == trainer.best_epoch)
    results_val[name] = {k: v for k, v in best.items() if k not in ("epoch", "train_loss")}
    checkpoints[name] = (build_clip, {k: v.cpu() for k, v in trainer.best_state.items()})
    print("best val:", fmt(results_val[name]))
    del model, trainer; _ = gc.collect()
    return results_val[name]

def load_trained(name) -> RetrievalCLIP:
    """(Provided) Rebuild a trained run from its checkpoint (best validation epoch)."""
    build_clip, state = checkpoints[name]
    model = RetrievalCLIP(build_clip()).to(device)
    model.load_state_dict({k: v.to(device) for k, v in state.items()}, strict=False)
    return model

projection_clip = lambda: setup_projection_peft(fresh_clip())

run("Proj InfoNCE lr=1e-5", projection_clip, info_nce_loss, epochs=10, lr=1e-5)
run("Proj Triplet lr=1e-5", projection_clip, triplet_loss, epochs=10, lr=1e-5)
plot_histories({k: histories[k] for k in ["Proj InfoNCE lr=1e-5", "Proj Triplet lr=1e-5"]},
               title="Projection-only: InfoNCE vs Triplet")

### 9.3 Learning-rate sweep (projection-only, InfoNCE) (Provided)

In [ ]:
# patience=None: run all 10 epochs so the full curves can be compared
for lr in [1e-6, 1e-4]:
    run(f"Proj InfoNCE lr={lr:.0e}", projection_clip, info_nce_loss, epochs=10, lr=lr, patience=None)

lr_runs = [k for k in histories if k.startswith("Proj InfoNCE lr=")]
plot_histories({k: histories[k] for k in lr_runs}, title="Projection-only InfoNCE: learning-rate sweep")

# choose the learning rate on VALIDATION (best-epoch mean_R)
best_lr_run = max(lr_runs, key=lambda k: results_val[k]["mean_R"])
BEST_LR = float(best_lr_run.split("lr=")[1])
print("Best learning rate on validation:", BEST_LR, f"(val mean_R={results_val[best_lr_run]['mean_R']:.4f})")

### 9.4 Overfitting: train longer (Provided)

In [ ]:
# patience=None: run all 25 epochs to see what happens after the best epoch (diagnostic run only)
run("Proj InfoNCE long", projection_clip, info_nce_loss, epochs=25, lr=BEST_LR, patience=None)
plot_histories({"Proj InfoNCE long": histories["Proj InfoNCE long"]}, metric="T2I_R@1",
               title="Projection-only, 25 epochs: training loss vs validation T2I R@1")

### 9.5 LoRA on the text encoder (Provided; the slowest step)

In [ ]:
LORA_EPOCHS = 5   # reduce to 3 if your laptop is slow
LORA_LR = 1e-4    # LoRA adapters usually need a larger learning rate than full weights

for r in [8, 2, 32]:   # alpha = 2r keeps alpha/r fixed
    run(f"LoRA r={r}", lambda r=r: setup_lora_peft(fresh_clip(), r=r, alpha=2 * r), info_nce_loss,
        epochs=LORA_EPOCHS, lr=BEST_LR, lora_lr=LORA_LR)

lora_runs = [k for k in histories if k.startswith("LoRA")]
plot_histories({k: histories[k] for k in lora_runs}, title="LoRA rank sweep (InfoNCE)")
for r in [2, 8, 32]:
    print(f"r={r:>2}: LoRA parameters = {24 * r * (512 + 512):>9,}")

# choose the rank on VALIDATION
BEST_LORA = max(lora_runs, key=lambda k: results_val[k]["mean_R"])
print("Best LoRA run on validation:", BEST_LORA, f"(val mean_R={results_val[BEST_LORA]['mean_R']:.4f})")

### 9.6 Merge the LoRA weights (Provided)

In [ ]:
# Uses the VALIDATION set: the test set stays unseen until 9.7.
lora_model = load_trained(BEST_LORA)
before = evaluate(lora_model, val_loader)
t0 = time.time(); encode_split(lora_model, val_loader); t_unmerged = time.time() - t0

n = merge_lora(lora_model)
after = evaluate(lora_model, val_loader)
t0 = time.time(); encode_split(lora_model, val_loader); t_merged = time.time() - t0

print(f"Merged {n} LoRA layers of {BEST_LORA}.")
print("val before merge:", fmt(before))
print("val after  merge:", fmt(after))
print(f"validation encoding time: unmerged {t_unmerged:.2f}s  |  merged {t_merged:.2f}s")
assert all(abs(before[k] - after[k]) < 2e-3 for k in before), "merging changed the results!"

### 9.7 Final comparison on the test set (Provided)

First the validation summary of **all** runs (used for the choices above), then the **single** test evaluation of the selected model of each method: zero-shot, projection-only with triplet loss, projection-only with InfoNCE at the learning rate chosen on validation, and LoRA with the rank chosen on validation (merged weights). The 25-epoch run is a diagnostic and is not a candidate.

In [ ]:
keys = ["I2T_R@1", "I2T_R@5", "I2T_R@10", "T2I_R@1", "T2I_R@5", "T2I_R@10", "mean_R"]
def print_table(title, table):
    print(title)
    print(f"{'method':<44}" + "".join(f"{k:>10}" for k in keys))
    for name, m in table.items():
        print(f"{name:<44}" + "".join(f"{m[k]:>10.4f}" for k in keys))
    print()

print_table("VALIDATION (best epoch of every run)", results_val)

results_test: Dict[str, Dict[str, float]] = {}
zero_shot = RetrievalCLIP(setup_projection_peft(fresh_clip())).to(device)
results_test["Zero-shot"] = evaluate(zero_shot, test_loader)
del zero_shot
for label, name in [("Projection-only, triplet", "Proj Triplet lr=1e-5"),
                    (f"Projection-only, InfoNCE (lr={BEST_LR:.0e})", best_lr_run)]:
    model = load_trained(name)
    results_test[label] = evaluate(model, test_loader)
    del model
results_test[f"{BEST_LORA} + projections (merged)"] = evaluate(lora_model, test_loader)
_ = gc.collect()

print_table("TEST (selected models, evaluated once)", results_test)

### 9.8 Qualitative retrieval (Provided)

In [ ]:
@torch.no_grad()
def show_retrievals(model: RetrievalCLIP, image_query: int = 0, text_query: int = 0, k: int = 3):
    """Positions refer to the test split. ✓ marks the ground-truth match."""
    img, txt = encode_split(model, test_loader)
    rows = test_ds.indices
    sim = img @ txt.T

    # Image -> text
    top = sim[image_query].topk(k).indices.tolist()
    plt.figure(figsize=(3, 3)); plt.imshow(all_data[rows[image_query]]["image"]); plt.axis("off")
    plt.title(f"Image query #{image_query}"); plt.show()
    print("True caption:", all_captions[rows[image_query]])
    for rank, j in enumerate(top, 1):
        print(f"  {rank}. {'✓' if j == image_query else ' '} ({sim[image_query, j]:.3f}) {all_captions[rows[j]]}")

    # Text -> image
    top = sim[:, text_query].topk(k).indices.tolist()
    print(f"\nText query #{text_query}: {all_captions[rows[text_query]]}")
    fig, axes = plt.subplots(1, k, figsize=(3 * k, 3))
    for ax, j in zip(axes, top):
        ax.imshow(all_data[rows[j]]["image"]); ax.axis("off")
        ax.set_title(f"{'✓ ' if j == text_query else ''}{sim[j, text_query]:.3f}")
    plt.show()

# Find one success and one failure for the selected LoRA model (merged weights) on text -> image R@1
img, txt = encode_split(lora_model, test_loader)
t2i_top1 = (img @ txt.T).argmax(dim=0).cpu()
correct = (t2i_top1 == torch.arange(len(t2i_top1)))
success, failure = int(correct.nonzero()[0]), int((~correct).nonzero()[0])
show_retrievals(lora_model, image_query=success, text_query=success)
show_retrievals(lora_model, image_query=failure, text_query=failure)

### 9.9 Analysis Questions

**Q1 [2 Marks] Performance comparison.** Using the table in 9.7, compare zero-shot, projection-only (InfoNCE and triplet) and LoRA. Which method improves Recall@K the most, in which direction (I2T vs T2I), and why?

**Your answer:**

_(write your answer here)_

**Q2 [1 Mark] InfoNCE vs triplet.** Compare the training-loss and validation curves of the two objectives (9.2). Which converges more smoothly, and which retrieves better? Explain using how many negatives each loss uses per example.

**Your answer:**

_(write your answer here)_

**Q3 [1 Mark] Learning rate.** Summarise how 1e-6, 1e-5 and 1e-4 change the loss curves and validation recall (9.3). Which learning rate was selected and why do we select it on the validation set?

**Your answer:**

_(write your answer here)_

**Q4 [1 Mark] Overfitting.** From the 25-epoch run (9.4), is there overfitting? At which epoch would you stop? With `PATIENCE = 3`, at which epoch would training have stopped, and how many epochs would that save?

**Your answer:**

_(write your answer here)_

**Q5 [2 Marks] LoRA.** (a) How many parameters does LoRA add for r = 2, 8, 32, and how does that compare with projection-only and with full fine-tuning? Derive the count. (b) How does the rank affect validation/test recall and training time? (c) Why is B initialised to zero, and what does merging (9.6) achieve?

**Your answer:**

_(write your answer here)_

**Q6 [1 Mark] Qualitative analysis.** Using the success and failure examples in 9.8 (you may choose other indices), describe what the model got right and suggest why it failed in the unsuccessful case.

**Your answer:**

_(write your answer here)_

### Bonus (not marked): LoRA on the image side

Apply LoRA to `q_proj`/`v_proj` of the **vision** encoder as well (`inject_lora(model.vision_model, ...)`). Note that this requires running the vision encoder live (the cached `vision_feats` can no longer be used), which is slow without a GPU. Does adapting the image side help more or less than the text side?

---

## AI Usage

List every generative AI tool you used for this notebook: which tool, what you asked it (the prompt or a short summary), and which part of the notebook it was used for. Write "None" if you did not use any.

| Tool | What you asked | Where it was used |
|---|---|---|
| | | |